# Skin Disease Detection using Deep Learning (HAM10000)

Trains and compares a CNN built from scratch and ResNet models on the HAM10000 dermoscopy dataset (7 classes), then exports the best ResNet for the Flask app.

**How to use:** run the cells top to bottom. Keep `QUICK = True` first (a ~3 minute dry run on a small subset that only checks the pipeline). Then set `QUICK = False`, use *Restart & Run all*, and wait for the real run.

| Syllabus module | Where it appears |
|---|---|
| 2.1 Loss functions, softmax | Cross-entropy, weighted cross-entropy (Experiment 5) |
| 2.2 Optimization | SGD vs momentum vs Adam vs RMSProp (Experiment 3) |
| 2.3 Regularization | Augmentation, dropout, weight decay, batch norm, early stopping (Experiment 4) |
| 4.1 CNN | Small CNN from scratch (Experiment 1) |
| 4.2 ResNet | ResNet-18 from scratch and pretrained (Experiment 2) |

## Step 1 - Setup
Clones the project code and checks that a GPU is available.

In [ ]:
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/atharvatawde69/skin_disease"
QUICK = True   # True = quick dry run on a small subset. Set to False for the real run.

if not Path("src").exists():
    if not Path("skin_disease").exists():
        !git clone {REPO_URL}
    %cd skin_disease
sys.path.insert(0, os.getcwd())

import torch
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable a GPU in the notebook settings before the real run")

## Step 2 - Get the dataset
On **Kaggle** the dataset is attached with *Add Input* (nothing to download). On **Colab** it is downloaded with the Kaggle API (you upload `kaggle.json` once).

In [ ]:
IN_KAGGLE = Path("/kaggle/input").exists()
if IN_KAGGLE:
    DATA_DIR = "/kaggle/input"
else:  # Google Colab
    DATA_DIR = "/content/ham10000"
    if not Path(DATA_DIR).exists():
        from google.colab import files
        print("Upload kaggle.json (Kaggle > Settings > API > Create New Token)")
        files.upload()
        !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
        !kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p {DATA_DIR} --unzip
print("data dir:", DATA_DIR)

## Step 3 - Look at the data
Two things matter here: the classes are **very imbalanced** (nevus `nv` is ~67% of the images), and one lesion can have **several photos**.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from src.data import load_metadata, split_by_lesion, compute_class_weights, HamDataset
from src.labels import CLASS_CODES, CLASS_NAMES

df = load_metadata(DATA_DIR)
if QUICK:
    df = df.sample(1500, random_state=0)
print(len(df), "images from", df.lesion_id.nunique(), "lesions")

counts = df.dx.value_counts()
ax = counts.plot.bar(title="Images per class")
ax.set_ylabel("images")
plt.show()
print((counts / counts.sum()).round(3).to_string())

In [ ]:
fig, axes = plt.subplots(2, 7, figsize=(18, 5))
for a in axes.ravel():
    a.axis("off")
for j, code_ in enumerate(CLASS_CODES):
    for i, (_, r) in enumerate(df[df.dx == code_].head(2).iterrows()):
        axes[i, j].imshow(Image.open(r.path))
        if i == 0:
            axes[i, j].set_title(f"{code_}\n{CLASS_NAMES[code_][:24]}", fontsize=8)
plt.show()

## Step 4 - Train / validation / test split (by lesion)
If photos of one lesion were split across train and test, the model would be tested on images it has effectively already seen. So we split on `lesion_id`: all photos of a lesion go to the same set. The split function asserts there is no overlap.

In [ ]:
df = split_by_lesion(df)
print(df.groupby("split").size().to_string())
print()
print(pd.crosstab(df.dx, df.split).to_string())

## Step 5 - Build datasets
Images are resized to 224x224 once and kept in RAM (about 1-2 minutes), so every epoch afterwards is fast. **Class weights** are inverse class frequencies: they make a mistake on a rare class cost more in the loss.

In [ ]:
IMG = 224
train_df, val_df, test_df = (df[df.split == s] for s in ("train", "val", "test"))
train_ds, val_ds, test_ds = (HamDataset(d, IMG) for d in (train_df, val_df, test_df))
class_weights = compute_class_weights(train_df)
print("class weights:", dict(zip(CLASS_CODES, class_weights.numpy().round(2))))

## Step 6 - Experiments
`run(name, **changes)` trains one configuration (early stopping on validation macro-F1) and tests the best epoch.
The reference setup is **ResNet-18 pretrained + Adam + augmentation + dropout + weight decay + weighted loss**. Each ablation changes one thing.

In [ ]:
from src.train import Config, fit
from src.evaluate import results_table, plot_history, plot_confusion_matrix

EPOCHS = 2 if QUICK else 15
Path("outputs/figures").mkdir(parents=True, exist_ok=True)
results = {}

BASE = dict(model="resnet18", pretrained=True, optimizer="adam", lr=3e-4, weight_decay=1e-4,
            dropout=0.3, augment=True, class_weights=True, epochs=EPOCHS, patience=5)

def run(name, **changes):
    cfg = Config(name=name, **{**BASE, **changes})
    results[name] = fit(cfg, train_ds, val_ds, test_ds, class_weights)

def curves(names, title, fname):
    fig = plot_history({n: results[n]["history"] for n in names}, title)
    fig.savefig(f"outputs/figures/{fname}.png", dpi=150)
    plt.show()

### Experiment 1 - CNN from scratch (module 4.1)
4 blocks of Conv -> BatchNorm -> ReLU -> MaxPool, then global average pooling, dropout and a linear layer. This is the baseline.

In [ ]:
run("cnn_scratch", model="cnn", pretrained=False, dropout=0.5, lr=1e-3)

### Experiment 2 - ResNet-18 (module 4.2): from scratch vs pretrained
Same network, once with random weights and once starting from ImageNet weights (transfer learning). Residual connections let gradients flow through deep networks, and pretrained filters already detect edges and textures.

In [ ]:
run("resnet18_scratch", pretrained=False, lr=1e-3)
run("resnet18_main")
curves(["cnn_scratch", "resnet18_scratch", "resnet18_main"], "CNN vs ResNet-18 (scratch) vs ResNet-18 (pretrained)", "models_comparison")

### Experiment 3 - Optimizers (module 2.2)
Same model, four optimizers, each with a sensible learning rate. Adam is the `resnet18_main` run above.

In [ ]:
run("opt_sgd", optimizer="sgd", lr=1e-2)
run("opt_momentum", optimizer="momentum", lr=1e-2)
run("opt_rmsprop", optimizer="rmsprop", lr=1e-4)
curves(["resnet18_main", "opt_sgd", "opt_momentum", "opt_rmsprop"], "Optimizers (resnet18_main = Adam)", "optimizers")

### Experiment 4 - Regularization (module 2.3)
Remove one regularizer at a time and watch the gap between train and validation loss (overfitting). Early stopping is active in every run: see `best_epoch` in the results table.

In [ ]:
run("reg_no_augment", augment=False)
run("reg_no_dropout", dropout=0.0)
run("reg_no_weight_decay", weight_decay=0.0)
run("reg_none", augment=False, dropout=0.0, weight_decay=0.0)
curves(["resnet18_main", "reg_no_augment", "reg_no_dropout", "reg_no_weight_decay", "reg_none"], "Regularization ablation", "regularization")

### Experiment 5 - Loss function (module 2.1)
Plain cross-entropy vs class-weighted cross-entropy. Without weights, the model can score high accuracy by favouring `nv`, but it misses rare and dangerous classes such as melanoma. Compare **balanced accuracy**, **macro-F1** and **melanoma recall**.

In [ ]:
run("loss_unweighted", class_weights=False)
curves(["resnet18_main", "loss_unweighted"], "Weighted vs unweighted cross-entropy", "loss")

### Optional - ResNet-50
A deeper ResNet. Slower, so it is off by default.

In [ ]:
RUN_RESNET50 = False
if RUN_RESNET50:
    run("resnet50", model="resnet50", batch_size=32)

## Step 7 - Results

In [ ]:
table = results_table(results)
table.to_csv("outputs/results.csv", index=False)
table

The final model is the ResNet with the best **validation** macro-F1 (we choose on validation, never on the test set).

In [ ]:
cands = {k: v for k, v in results.items() if v["cfg"]["model"].startswith("resnet")}
best_name = max(cands, key=lambda k: cands[k]["best_val_f1"])
best = results[best_name]
print("selected:", best_name, "| val macro-F1", round(best["best_val_f1"], 4), "| test macro-F1", round(best["test"]["macro_f1"], 4))

fig = plot_confusion_matrix(best["test"]["confusion_matrix"], title=f"{best_name} - test set (rows = true class)")
fig.savefig("outputs/figures/confusion_matrix.png", dpi=150)
plt.show()
pd.DataFrame(best["test"]["per_class"]).T.round(3)

### Grad-CAM: what is the model looking at?
Red areas pushed the model most towards its prediction. A good model should focus on the lesion, not on the background or a ruler.

In [ ]:
from src.models import build_model, gradcam_target_layer
from src.gradcam import GradCAM, overlay_cam
from src.data import build_transform

ckpt = torch.load(best["ckpt"], map_location="cpu", weights_only=True)
model = build_model(ckpt["cfg"]["model"], pretrained=False, dropout=ckpt["cfg"]["dropout"])
model.load_state_dict(ckpt["model"])
model.eval()
cam = GradCAM(model, gradcam_target_layer(model))
tf = build_transform(IMG, augment=False)

sample = test_df.sample(8, random_state=1)
fig, axes = plt.subplots(2, 8, figsize=(22, 6))
for a in axes.ravel():
    a.axis("off")
for k, (_, r) in enumerate(sample.iterrows()):
    img = Image.open(r.path).convert("RGB").resize((IMG, IMG))
    heat, idx, probs = cam(tf(img).unsqueeze(0))
    axes[0, k].imshow(img)
    axes[0, k].set_title(f"true: {r.dx}", fontsize=9)
    axes[1, k].imshow(overlay_cam(img, heat))
    axes[1, k].set_title(f"pred: {CLASS_CODES[idx]} ({probs[idx]:.2f})", fontsize=9)
fig.savefig("outputs/figures/gradcam.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 8 - Export the model for the web app

In [ ]:
import shutil
Path("models").mkdir(exist_ok=True)
shutil.copy(best["ckpt"], "models/best.pt")
shutil.make_archive("skin_disease_outputs", "zip", "outputs")
print("saved models/best.pt and skin_disease_outputs.zip")
if QUICK:
    print("QUICK mode: this model is only a pipeline check. Set QUICK = False and re-run everything for the real model.")
if not IN_KAGGLE:
    from google.colab import files
    files.download("models/best.pt")
    files.download("skin_disease_outputs.zip")